# 🛰️ Nova Neural Fusion: Cloud Training (SUPER SAFE EDITION)
Este cuaderno tiene **Descarga Automática** y **Respaldo en Google Drive**.

### 🚀 Instrucciones:
1. Sube `art_universe_500.jsonl` a la izquierda.
2. (Opcional pero recomendado) Ejecuta la celda de Google Drive para que no se pierda nada.
3. Dale a **Entorno de ejecución > Ejecutar todo**.

In [ ]:
# 1. Instalar herramientas
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. BLOQUE DE SEGURIDAD: Montar Google Drive (Opcional)
from google.colab import drive
import os
try:
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/Nova_Backups/"
    os.makedirs(SAVE_PATH, exist_ok=True)
    print("✅ Google Drive conectado. Todo se guardara tambien alli.")
except:
    SAVE_PATH = "/content/"
    print("⚠️ Drive no conectado. Solo se usara descarga al finalizar.")

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-1b-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

dataset = load_dataset("json", data_files="art_universe_500.jsonl", split="train")
def formatting_prompts_func(examples):
    texts = [f"### Instruction:\n{i}\n\n### Response:\n{o}" for i, o in zip(examples["instruction"], examples["output"])]
    return { "text" : texts, }
dataset = dataset.map(formatting_prompts_func, batched = True)

model = FastLanguageModel.get_peft_model(model, r = 16, lora_alpha = 16, random_state = 3407)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    dataset_text_field = "text", max_seq_length = 2048,
    args = TrainingArguments(max_steps = 300, learning_rate = 2e-4, output_dir = "outputs")
)
trainer.train()

In [ ]:
# 5. EXPORTACION Y DESCARGA BLINDADA
MODEL_NAME = "nova_ultra_500_artists"
model.save_pretrained_gguf(MODEL_NAME, tokenizer, quantization_method = "q4_k_m")

# Guardar en Drive si esta disponible
import shutil
final_file = MODEL_NAME + "-unsloth.Q4_K_M.gguf"
if SAVE_PATH != "/content/":
    shutil.copy(final_file, SAVE_PATH + final_file)
    print(f"📦 Copia de seguridad guardada en Drive: {SAVE_PATH}")

# Descarga automatica al navegador
from google.colab import files
files.download(final_file)